# STOFS-3D Coastal Boundary Integration

This notebook demonstrates how to use the `CoastalBoundary` class to download NOAA STOFS-3D-Atlantic coastal storm surge forecasts and write stage boundary conditions for HEC-RAS models.

**STOFS-3D-Atlantic** (Surge, Tide, and Overland Flow with 3D hydrodynamics) is NOAA's operational coastal storm surge forecast model:
- **Coverage**: U.S. Atlantic Coast, Gulf of Mexico, and Caribbean
- **Mesh**: Unstructured triangular mesh (~80 m nearshore to ~50 km offshore)
- **Forecast horizon**: 48 hours
- **Vertical datum**: NAVD88 (meters)
- **Cycles**: 00z, 06z, 12z, 18z UTC (4 cycles per day)
- **Data source**: NOAA NOMADS server (`https://nomads.ncep.noaa.gov/pub/data/nccf/com/stofs/prod`)

> **Note**: Download and extraction steps require internet access to NOAA NOMADS servers. Code cells that require internet access will gracefully skip and show expected output.

In [ ]:
# =============================================================================
# DEVELOPMENT MODE TOGGLE
# =============================================================================
# Set USE_LOCAL_SOURCE based on your setup:
#   True  = Use local source code (for developers editing ras-commander)
#   False = Use pip-installed package (for users)
# =============================================================================

USE_LOCAL_SOURCE = True  # <-- TOGGLE THIS

# -----------------------------------------------------------------------------
if USE_LOCAL_SOURCE:
    import sys
    from pathlib import Path
    local_path = str(Path.cwd().parent)  # Parent of examples/ = repo root
    if local_path not in sys.path:
        sys.path.insert(0, local_path)  # Insert at position 0 = highest priority
    print(f"LOCAL SOURCE MODE: Loading from {local_path}/ras_commander")
else:
    print("PIP PACKAGE MODE: Loading installed ras-commander")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

from ras_commander.sources.federal import CoastalBoundary

## What You'll Learn

- **Download STOFS-3D forecasts** from NOAA NOMADS using `CoastalBoundary.download_stofs3d()`
- **Extract water surface elevation (WSE)** at any coastal point using `CoastalBoundary.extract_wse_at_point()`
- **Create stage boundary conditions** for HEC-RAS using `CoastalBoundary.generate_stage_bc()`
- **Handle datum conversions** from NAVD88 meters to feet with optional datum adjustments
- **Check forecast availability** before downloading with `CoastalBoundary.check_availability()`

## STOFS-3D Overview

STOFS-3D-Atlantic uses an **unstructured triangular mesh** that adapts spatial resolution to coastal complexity:

| Zone | Approximate Resolution |
|------|------------------------|
| Nearshore (< 20 m depth) | ~80 m |
| Inner shelf | ~500 m - 2 km |
| Outer shelf | ~5 km |
| Deep ocean | ~50 km |

**Output types**:
- `points` - Time series at selected output points (faster, ~50 MB per file)
- `fields` - Gridded spatial output (larger, used for flood mapping)

**Temporal resolution**: Hourly output for the full 48-hour forecast window

**Why it matters for HEC-RAS**: STOFS-3D provides physically consistent tidal and storm surge predictions that can be used directly as downstream stage boundary conditions, replacing manually estimated tidal curves or NOAA tidal predictions.

In [ ]:
# Get STOFS-3D model information
info = CoastalBoundary.get_info()
print("STOFS-3D-Atlantic Information:")
for key, value in info.items():
    print(f"  {key}: {value}")

## Example 1: Gulf Coast (Galveston Bay)

Galveston Bay is an excellent test case for STOFS-3D integration:
- Low tidal range (~1-2 ft) makes storm surge the dominant water level driver
- Well-documented historical surge events (Hurricane Harvey, Ike)
- Many HEC-RAS models use Galveston Bay as downstream boundary

### Step 1: Check Data Availability

In [ ]:
# Check whether the latest STOFS-3D cycle is available on NOMADS
try:
    available = CoastalBoundary.check_availability()
    print(f"Latest STOFS-3D cycle available: {available}")
except Exception as e:
    print(f"Availability check requires internet: {e}")

### Step 2: Download STOFS-3D Forecast

In [ ]:
# Create output directory
stofs_dir = Path("example_data/stofs3d_gulf")
stofs_dir.mkdir(parents=True, exist_ok=True)

try:
    stofs_files = CoastalBoundary.download_stofs3d(
        output_dir=stofs_dir,
        file_type="points"  # Point data (faster, smaller than fields)
    )
    print(f"Downloaded {len(stofs_files)} STOFS-3D files")
    for f in stofs_files:
        print(f"  {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")
except Exception as e:
    print(f"Download requires internet: {e}")
    print()
    print("Expected output:")
    print("  Downloaded 1-2 STOFS-3D NetCDF files")
    print("  stofs_3d_atl.t00z.points.cwl.nc (~50 MB)")
    stofs_files = []

### Step 3: Extract WSE at Galveston Bay Entrance

In [ ]:
# Galveston Bay entrance coordinates
galveston_lat = 29.35
galveston_lon = -94.77

try:
    wse_df = CoastalBoundary.extract_wse_at_point(
        stofs_dir=stofs_dir,
        lat=galveston_lat,
        lon=galveston_lon,
        units="feet"  # Convert from NAVD88 meters to feet
    )

    print(f"Extracted {len(wse_df)} time steps at ({galveston_lat}, {galveston_lon})")
    print(f"\nWSE Statistics (feet NAVD88):")
    print(f"  Min:  {wse_df['wse_ft'].min():.2f} ft")
    print(f"  Max:  {wse_df['wse_ft'].max():.2f} ft")
    print(f"  Mean: {wse_df['wse_ft'].mean():.2f} ft")
    print(f"\nFirst 10 time steps:")
    print(wse_df.head(10).to_string(index=False))
except Exception as e:
    print(f"Extraction requires downloaded data: {e}")
    print()
    print("Example output:")
    print("  datetime              wse_m     wse_ft")
    print("  2025-01-01 00:00:00   0.305     1.00")
    print("  2025-01-01 01:00:00   0.335     1.10")
    print("  2025-01-01 02:00:00   0.427     1.40")
    print("  ...")
    wse_df = None

### Step 4: Write Stage Boundary Condition

In [ ]:
# Write WSE as downstream stage boundary condition in HEC-RAS .u## file.
# CoastalBoundary.generate_stage_bc() writes directly to the unsteady file.

print("To write the stage boundary condition:")
print()
print("  CoastalBoundary.generate_stage_bc(")
print("      wse_timeseries=wse_df,")
print("      unsteady_file='MyProject.u01',")
print("      bc_location='Downstream',")
print("      units='feet',")
print("      datum_adjustment_ft=0.0  # Adjust if needed")
print("  )")
print()
print("This modifies the .u## file in-place:")
print("  - Finds 'Stage Hydrograph=' section for the BC location")
print("  - Writes values in HEC-RAS fixed-width format (8.2f, 10/line)")
print("  - Updates the Interval= line automatically")
print()
print("Datum Adjustment:")
print("  - STOFS-3D uses NAVD88 datum")
print("  - HEC-RAS model may use a different vertical datum")
print("  - Use datum_adjustment_ft to add/subtract offset")

## Example 2: Atlantic Coast (Charleston Harbor)

Charleston Harbor contrasts with Galveston Bay in an important way:
- High tidal range (~5-6 ft) means tides dominate normal water levels
- Complex tidal dynamics (semidiurnal pattern)
- Storm surge compounds on top of a larger tidal baseline

This comparison demonstrates how STOFS-3D handles both microtidal (Gulf) and mesotidal (Atlantic) environments.

In [ ]:
# Charleston Harbor coordinates
charleston_lat = 32.78
charleston_lon = -79.93

try:
    wse_charleston = CoastalBoundary.extract_wse_at_point(
        stofs_dir=stofs_dir,
        lat=charleston_lat,
        lon=charleston_lon,
        units="feet"
    )

    print(f"Charleston Harbor WSE ({charleston_lat}, {charleston_lon}):")
    print(f"  Time steps: {len(wse_charleston)}")
    print(f"  WSE range: {wse_charleston['wse_ft'].min():.2f} to {wse_charleston['wse_ft'].max():.2f} ft")
    print(f"  Tidal range: {wse_charleston['wse_ft'].max() - wse_charleston['wse_ft'].min():.2f} ft")
except Exception as e:
    print(f"Extraction requires data: {e}")
    print()
    print("Expected output for Charleston:")
    print("  Time steps: 48")
    print("  WSE range: -2.50 to 3.20 ft")
    print("  Tidal range: 5.70 ft (typical for Charleston)")
    wse_charleston = None

### Comparing Locations

In [ ]:
print("Comparison of Coastal Locations:")
print("=" * 60)
print(f"{'Feature':<25} {'Galveston Bay':<18} {'Charleston Harbor'}")
print("-" * 60)
print(f"{'Latitude':<25} {galveston_lat:<18} {charleston_lat}")
print(f"{'Longitude':<25} {galveston_lon:<18} {charleston_lon}")
print(f"{'Typical tidal range':<25} {'~1-2 ft':<18} {'~5-6 ft'}")
print(f"{'Storm surge risk':<25} {'High (hurricanes)':<18} {'Moderate'}")
print(f"{'Tidal pattern':<25} {'Diurnal':<18} {'Semidiurnal'}")
print(f"{'Datum':<25} {'NAVD88':<18} {'NAVD88'}")
print()
print("Key considerations:")
print("  - Gulf Coast: Lower tidal range but higher storm surge potential")
print("  - Atlantic Coast: Higher tidal range, complex tidal dynamics")
print("  - Both: NAVD88 meters in STOFS-3D, auto-converted to feet")

## Key Takeaways

### Summary

- **STOFS-3D-Atlantic** provides 48-hour coastal water level forecasts combining tides and storm surge
- **`CoastalBoundary`** class handles download, extraction, and BC generation in a simple API
- Data is in **NAVD88 meters**, auto-converted to feet via `METERS_TO_FEET = 3.28084`
- **Unstructured mesh**: extraction finds the nearest mesh node to your specified coordinates
- **Stage boundary conditions** are written directly to HEC-RAS `.u##` files in fixed-width format

### Best Practices

1. **Verify datum**: STOFS-3D uses NAVD88; check your HEC-RAS model's vertical datum before integrating
2. **Check mesh proximity**: The extraction finds the nearest node - verify the distance is acceptable for your application
3. **Forecast horizon**: Maximum 48 hours - plan your simulation window accordingly
4. **Cycle selection**: Choose the forecast cycle (`0`, `6`, `12`, or `18` UTC) that best fits your operational window
5. **File type**: Use `"points"` for single-location extraction (faster), `"fields"` only if you need spatial coverage

### Integration with HEC-RAS

The typical coastal workflow using `CoastalBoundary`:

```python
from ras_commander.sources.federal import CoastalBoundary
from ras_commander import init_ras_project, RasCmdr
from pathlib import Path

# 1. Download latest STOFS-3D forecast
stofs_dir = Path("stofs_data")
CoastalBoundary.download_stofs3d(stofs_dir, file_type="points")

# 2. Extract WSE at downstream boundary location
wse_df = CoastalBoundary.extract_wse_at_point(
    stofs_dir=stofs_dir,
    lat=29.35, lon=-94.77,
    units="feet"
)

# 3. Write stage BC to unsteady file
CoastalBoundary.generate_stage_bc(
    wse_timeseries=wse_df,
    unsteady_file="MyProject.u01",
    bc_location="Downstream Boundary"
)

# 4. Execute HEC-RAS
init_ras_project("MyProject", "6.6")
RasCmdr.compute_plan("01")
```

### Related Notebooks

- **915_realtime_forecast_workflow.ipynb** - Complete operational forecast pipeline (HRRR + STOFS-3D + HEC-RAS)
- **913_bc_generation_from_live_gauge.ipynb** - Upstream flow BC from live USGS gauges
- **900_aorc_precipitation.ipynb** - AORC historic precipitation for model calibration

## Cleanup

In [ ]:
import shutil
for d in ["example_data/stofs3d_gulf"]:
    if Path(d).exists():
        shutil.rmtree(d)
        print(f"Cleaned up: {d}")
print("Done!")